# F0：在九格世界里重新发明世界模型

这份 Notebook 不需要 GPU，也不使用神经网络。我们从一个会走进陷阱的机器人开始，依次加入状态转移、多步推演、任务评价、规划、从经历学习和现实修正。

每一节运行以前，先在纸上写下自己的判断。结果与判断不同的地方，正是本次实验最值得保留的部分。

In [ ]:
from pathlib import Path
import random
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'hwm').exists():
    ROOT = ROOT.parent
assert (ROOT / 'src' / 'hwm').exists(), '请从课程仓库内运行本 Notebook'
sys.path.insert(0, str(ROOT / 'src'))

from hwm.gridworld import (
    ACTION_SYMBOLS,
    EmpiricalDynamics,
    GridWorld,
    Transition,
    format_trajectory,
    lookahead,
    mpc_episode,
    rollout,
)

print(f'课程仓库：{ROOT}')
print('环境检查通过：本实验只使用 Python 标准库。')

## 1. 只朝终点走近

机器人位于左上角 `S`，终点是 `G`，`×` 表示陷阱，`■` 表示墙壁。

先不计算未来。只看哪个动作能让下一格离终点更近，我们会选择什么？

In [ ]:
world = GridWorld()
print(world.render(state=world.start))

greedy_action = world.greedy_action(world.start)
greedy_result = world.transition(world.start, greedy_action)
print(f'\n只看距离时选择：{greedy_action} {ACTION_SYMBOLS[greedy_action]}')
print(f'实际到达：{greedy_result.next_state}，奖励：{greedy_result.reward}')
assert greedy_result.next_state in world.traps

模型看见了终点，却没有计算动作会把机器人带到哪里。我们先不学习复杂算法，只写出一个接口：

```text
当前状态 + 候选动作 → 下一状态
```

九格世界的 `transition` 是一台人工写成的最小世界模型。

In [ ]:
print('同一个现在，替换四个动作：')
for action in ('right', 'down', 'up', 'left'):
    result = world.transition(world.start, action)
    print(
        f'{ACTION_SYMBOLS[action]} {action:>5} -> '
        f'{result.next_state}, reward={result.reward:>5.1f}, done={result.done}'
    )

这是一项最小反事实检查：起点、地图和规则都保持不变，只替换动作。若四个动作得到完全相同的未来，模型就没有真正使用动作。

## 2. 一步安全，不等于最后成功

向下不会立刻掉进陷阱。但接下来应该怎样走？我们把模型预测出的状态再次送回模型，连续推演六步。

In [ ]:
safe_actions = ('down', 'down', 'right', 'right', 'up', 'up')
imagined = rollout(world, world.start, safe_actions)

print('尚未真实执行的路线：')
print(format_trajectory(imagined))
print(f'总回报：{sum(item.reward for item in imagined)}')
print('\n路线经过的地图：')
print(world.render(path=[item.next_state for item in imagined]))

assert imagined[-1].next_state == world.goal

连续调用模型得到的想象轨迹称为 **rollout**。模型现在可以产生多条未来，却还不知道哪条更好。

九格世界规定：普通移动为 `-1`，陷阱为 `-10`，终点为 `+10`。路线中所有奖励相加，得到回报。

In [ ]:
trap_route = rollout(world, world.start, ('right',))
goal_route = rollout(world, world.start, safe_actions)

rows = [
    ('走进陷阱', sum(item.reward for item in trap_route), len(trap_route)),
    ('绕路到终点', sum(item.reward for item in goal_route), len(goal_route)),
]
print(f"{'路线':<12}{'回报':>8}{'步数':>8}")
for name, total_return, steps in rows:
    print(f'{name:<12}{total_return:>8.1f}{steps:>8}')

世界模型回答“会发生什么”，奖励说明“当前任务在意什么”。改变终点以后，移动规律不需要重新学习，路线的好坏却会改变。

## 3. 规划深度改变选择

规划器会枚举短动作序列，调用世界模型得到路线，再选择预测回报最高的一条。深度越大，能看见更远的结果，搜索量也会迅速增长。

In [ ]:
planning_world = GridWorld(
    start=(0, 0),
    goal=(2, 2),
    traps=((0, 2),),
    walls=((1, 1),),
)
print('这一次，陷阱放在向右两步的位置：')
print(planning_world.render(state=planning_world.start))
print(f"\n{'深度':>4} {'第一动作':>8} {'预测回报':>10} {'候选序列':>10}")
plans = {}
for depth in (1, 3, 6):
    plan = lookahead(
        planning_world,
        planning_world.start,
        depth,
        action_order=('right', 'down', 'up', 'left'),
    )
    plans[depth] = plan
    print(
        f'{depth:>4} {ACTION_SYMBOLS[plan.action] + plan.action:>8} '
        f'{plan.predicted_return:>10.1f} {plan.evaluated_sequences:>10}'
    )

assert plans[1].action == 'right'
assert plans[6].action == 'down'

深度 1 只看见右边离终点近。深度 6 能看见完整绕行路线，但它检查了 $4^6=4096$ 组动作。连续动作或更长 horizon 无法这样全部枚举，后续才会需要 CEM、MCTS 或 Actor。

## 4. 模型只想多步，环境只走一步

真实环境可能打滑。我们让规划器在确定模型中想六步，但每次只在带打滑的环境中执行第一步。读取真实位置以后，再重新规划。

In [ ]:
slippery_world = GridWorld(slip_probability=0.25)
real_steps, replans = mpc_episode(
    slippery_world,
    model=world,
    depth=6,
    max_steps=14,
    seed=7,
)

print('每一步都重新读取真实位置：')
for index, (step, plan) in enumerate(zip(real_steps, replans), start=1):
    predicted = world.next_state(step.state, plan.action)
    marker = '修正' if predicted != step.next_state else '一致'
    print(
        f'{index:>2}. {step.state} {ACTION_SYMBOLS[plan.action]} '
        f'预测 {predicted}，真实 {step.next_state} [{marker}]'
    )

print(f'\n最后到达：{real_steps[-1].next_state}')
assert real_steps[-1].next_state == slippery_world.goal

这种“预测多步、只执行一步、再观察”的节奏就是模型预测控制（MPC）的基本形式。它没有假设模型永远正确，而是不断用现实修正想象。

## 5. 拿走人写的转移表

到目前为止，移动规律由 `GridWorld.next_state` 写好。现实没有附送这个函数。下面从三条经历中学习“左下角向右”会发生什么。

In [ ]:
samples = [
    Transition((2, 0), 'right', -1.0, (2, 1), False),
    Transition((2, 0), 'right', -1.0, (2, 1), False),
    Transition((2, 0), 'right', -1.0, (2, 0), False),
]
learned_model = EmpiricalDynamics().fit(samples)
distribution = learned_model.distribution((2, 0), 'right')

print('从三次经历学到的结果分布：')
for next_state, probability in distribution.items():
    print(f'P({next_state} | (2, 0), right) = {probability:.3f}')

assert abs(sum(distribution.values()) - 1.0) < 1e-9

同一个动作出现了两种结果。模型没有输出两个位置的平均值，而是保留两种可能及其概率。

现在查看一个数据中从未出现的状态—动作。

In [ ]:
known = learned_model.distribution((2, 0), 'right')
unknown = learned_model.distribution((1, 2), 'up')
print(f'见过的转移：{known}')
print(f'没见过的转移：{unknown}')
assert known and not unknown

空字典不是一个好预测，却是一份重要信息：数据没有告诉模型这里会发生什么。复杂神经网络通常仍会输出一个数字，所以我们还需要 OOD 检查和不确定性校准，防止“不知道”被伪装成肯定答案。

## 6. 三个接口重新放好

完成以上实验后，我们已经亲手得到三个不同程序：

```text
世界模型：当前状态 + 候选动作 → 可能的未来
规划器：调用世界模型比较几种未来 → 选择动作
策略：当前信息 → 直接输出动作
```

本 Notebook 的规划器没有训练策略。第 2 章会先用 PlaNet/CEM 每次现场搜索，再用 Dreamer 把想象中的好动作学成 Actor。

## 小结

- [ ] 我能解释为什么只朝终点走近会掉入陷阱。
- [ ] 我能写出 `state + action -> next_state` 的最小接口。
- [ ] 我知道 rollout、reward、return 和 Planner 分别补上什么缺口。
- [ ] 我能区分世界模型、规划器和策略。
- [ ] 我知道同一动作可能对应多个未来。
- [ ] 我知道 MPC 为什么只执行计划的第一步。

## 作业

1. 修改地图，使深度为 3 的规划失败、深度为 6 的规划成功。
2. 把终点移到左下角。不要修改转移模型，检查最优动作怎样变化。
3. 收集至少 30 条带打滑的 transition，比较经验分布与真实打滑概率。
4. 设计一张地图，使一个错误模型提供不存在的捷径。说明更深搜索为什么可能更糟。

完成后，请保留一张失败地图和一句说明：原来的办法缺少什么，下一步准备怎样检验。

## 参考资料

- [World Models](https://arxiv.org/abs/1803.10122)：把视觉、记忆和控制接进可学习的梦境。
- [Learning to Think](https://doi.org/10.1016/B978-1-55860-141-3.50030-4)：Dyna 的早期论文。
- [PlaNet](https://arxiv.org/abs/1811.04551)：在 latent dynamics 中用 CEM 规划。
- [Dreamer](https://arxiv.org/abs/1912.01603)：在 latent imagination 中学习行为。